In [ ]:
import pandas as pd
import re
import html
from sklearn.feature_extraction.text import TfidfVectorizer

# Load the raw training data
df_train = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")

# Define the universal cleaning function
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")

def clean_universal(text: str) -> str:
    if not isinstance(text, str):
        return ""
    s = html.unescape(text) # Handles HTML entities like &amp;
    s = URL_RE.sub(" <URL> ", s) # Replaces URLs
    s = MENTION_RE.sub(" <USER> ", s) # Replaces user mentions
    s = HASHTAG_RE.sub(r" \1 ", s) # Removes '#' but keeps the word
    s = s.replace("&amp;", "and")
    s = re.sub(r"\s+", " ", s).strip()
    s = s.lower() # Converts to lowercase
    return s

print("--- 1. Visualizing Universal Preprocessing ---\\n")

# Take a sample of 20 tweets for visualization
sample_df = df_train.sample(20, random_state=42)

# Create 'Before' and 'After' columns
before_texts = sample_df['text'].tolist()
after_texts = [clean_universal(text) for text in before_texts]

comparison_df = pd.DataFrame({
    'Before': before_texts,
    'After': after_texts
})

# To display the full text in the output, we can adjust pandas display options
pd.set_option('display.max_colwidth', None)

# Display the comparison DataFrame
print("Side-by-side comparison of 20 tweets before and after cleaning:")
display(comparison_df)

print("\\n" + "="*50 + "\\n")

# --- 2. Visualizing the TF-IDF Matrix ---

# Take a smaller sample of the cleaned text for the matrix visualization
tfidf_sample = comparison_df['After'].head(10).tolist()

# Initialize and fit the TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)
tfidf_matrix = tfidf_vectorizer.fit_transform(tfidf_sample)
feature_names = tfidf_vectorizer.get_feature_names_out()

print("--- 2. Learned Vocabulary (Features) ---")
print(f"The vectorizer found {len(feature_names)} unique terms that meet the criteria.")
print(list(feature_names))
print("\\n" + "="*50 + "\\n")


# Create and display the TF-IDF matrix DataFrame
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names)

print("--- 3. Resulting TF-IDF Matrix ---")
print("Each tweet (row) is now represented by a vector of scores for each term (column).")
display(df_tfidf)


## 🧩 Understanding the TF–IDF Matrix

The TF–IDF (Term Frequency–Inverse Document Frequency) transformation converts text data into numerical feature vectors that capture how important each word is to each document **relative to the entire dataset**.

---

### 1️⃣ What TF–IDF Does

Each **row** in the matrix represents one document (e.g., a tweet).  
Each **column** represents one term (word) from the learned vocabulary.  

Each cell’s value is the **TF–IDF score**, which measures:

> How important a word is within a document (Term Frequency)  
> weighted by how rare that word is across all documents (Inverse Document Frequency).

---

### 2️⃣ Learned Vocabulary

Example:


In [ ]:
['and', 'can', 'for', 'got', 'in', 'is', 'it', 'like', 'my', 'of',
'that', 'the', 'this', 'this is', 'to', 'url', 'user', 'want', 'want to', 'you']

This is your **feature space** — the 20 distinct tokens the vectorizer kept after text preprocessing and vocabulary size limits.  
Each token becomes a column in the TF–IDF matrix.

---

### 3️⃣ Interpreting the Matrix

| Row | and | can | for | got | in | ... | you |
|-----|-----|-----|-----|-----|----|-----|-----|
| 0 | 0.000 | 0.526 | 0.000 | 0.000 | 0.000 | ... | 0.601 |
| 1 | 0.286 | 0.000 | 0.622 | 0.367 | 0.286 | ... | 0.000 |

- Each value (e.g., `0.526101`) is the **TF–IDF score** for that term in that document.  
- Higher values → the word is **more important** in that specific text.  
- `0` → the word doesn’t appear or is not informative for that document.

Examples:
- Row 0 has high scores for **“can”** (0.526) and **“you”** (0.601) → key words for that tweet.  
- Row 1 has high scores for **“for”** (0.622) and **“got”** (0.367).

---

### 4️⃣ Why It Matters

The TF–IDF matrix is what you feed into machine learning models (e.g., Logistic Regression, SVM, or Neural Networks) instead of raw text.

This gives your model a numeric representation of how relevant each term is to a specific sample.

- Each document → a *vector* of TF–IDF scores.  
- Each dimension → the weighted importance of one vocabulary word.

---

### 5️⃣ Key Intuitions

| Concept | Meaning | Effect |
|----------|----------|--------|
| **Term Frequency (TF)** | How often the term appears in a document | Common words within a document have higher TF |
| **Inverse Document Frequency (IDF)** | How rare the term is across all documents | Common words across all documents have lower IDF |
| **TF × IDF** | Combined importance | Highlights words that are frequent *and* distinctive |

So, TF–IDF downweights frequent but uninformative words (like “the”, “is”)  
and emphasizes uncommon yet meaningful ones.

---

### 🧠 TL;DR
- Rows = documents  
- Columns = words  
- Values = importance scores  
- Use these vectors as numeric input features for your ML model.